# Эксперимент 24: Phoneme-level HMM/GMM pipeline

**Статья:** Dysarthria speech recognition by phonemes using hidden Markov models (Распознавание дизартрической речи по фонемам с использованием скрытых марковских моделей) 2014

**Ссылка:** [https://moitvivt.ru/ru/journal/pdf?id=1471](https://moitvivt.ru/ru/journal/pdf?id=1471)

**Краткое описание модели:** Класс-условные акустические модели последовательностей MFCC (GMM-HMM-приближение без forced alignment) с вероятностным decision rule и fusion буквенных признаков.

**Содержание статьи:** Вместо агрегированных MFCC+SVM реализован вероятностный sequence-level pipeline для каждого класса, приближенный к статистической логике фонемных HMM-подходов, с отбором конфигурации по валидации.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [ ]:
def extract_mfcc_seq(path, n_mfcc=20, max_frames=320):
    y, sr = data_utils.load_audio(path, sr=config.TARGET_SR, max_sec=config.MAX_DURATION_SEC)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=config.HOP_LENGTH, n_fft=config.WIN_LENGTH).T
    if mfcc.shape[0] > max_frames:
        idx = np.linspace(0, mfcc.shape[0] - 1, max_frames).astype(int)
        mfcc = mfcc[idx]
    return mfcc.astype(np.float32)

seq_train = [extract_mfcc_seq(p) for p in paths_train]
seq_val   = [extract_mfcc_seq(p) for p in paths_val]
seq_test  = [extract_mfcc_seq(p) for p in paths_test]

## 3. Обучение, оценка и сохранение результата

In [ ]:
def fit_gmm_for_class(seqs, n_components, cov_type):
    X = np.vstack(seqs)
    gmm = GaussianMixture(n_components=n_components, covariance_type=cov_type, reg_covar=1e-4, random_state=config.RANDOM_STATE, max_iter=200)
    gmm.fit(X)
    return gmm

def avg_loglike(gmm, seq):
    return float(np.mean(gmm.score_samples(seq)))

cands = [(4, "diag"), (8, "diag"), (12, "diag"), (8, "full")]
best = None
best_f1 = -1.0
for n_comp, cov_type in cands:
    gmm_good = fit_gmm_for_class([s for s, y in zip(seq_train, y_train) if y == config.CLASS_GOOD], n_comp, cov_type)
    gmm_bad  = fit_gmm_for_class([s for s, y in zip(seq_train, y_train) if y == config.CLASS_BAD], n_comp, cov_type)
    val_scores = np.array([[avg_loglike(gmm_good, s), avg_loglike(gmm_bad, s)] for s in seq_val])
    val_pred = (val_scores[:, 1] > val_scores[:, 0]).astype(int)
    val_f1 = f1_score(y_val, val_pred, average="macro")
    if val_f1 > best_f1:
        best_f1 = val_f1
        best = (n_comp, cov_type, gmm_good, gmm_bad)

n_comp, cov_type, gmm_good, gmm_bad = best

def score_set(seqs):
    sc = np.array([[avg_loglike(gmm_good, s), avg_loglike(gmm_bad, s)] for s in seqs])
    delta = sc[:, 1] - sc[:, 0]
    p_bad = 1.0 / (1.0 + np.exp(-delta))
    return p_bad

proba_train = score_set(seq_train)
proba_test = score_set(seq_test)

lr_letters = LogisticRegression(class_weight="balanced", max_iter=2000, solver="liblinear", random_state=config.RANDOM_STATE)
lr_letters.fit(letters_train, y_train)
proba_train_l = lr_letters.predict_proba(letters_train)[:, 1]
proba_test_l = lr_letters.predict_proba(letters_test)[:, 1]
proba_test = 0.8 * proba_test + 0.2 * proba_test_l

y_pred = (proba_test >= 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, proba_test)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_24_hmm_proxy", 
    experiment_name="Phoneme-level HMM/GMM", 
    model="Class-conditional GMM sequence model", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"gmm_hmm_like/n_comp={n_comp},cov={cov_type}", 
    num_params=None, 
    train_time_sec=None
)

              precision    recall  f1-score   support

        good       0.80      0.68      0.73       282
         bad       0.49      0.64      0.55       135

    accuracy                           0.67       417
   macro avg       0.64      0.66      0.64       417
weighted avg       0.70      0.67      0.68       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.666667,0.64364,0.553055,0.733727,0.488636,0.637037


Результат сохранён в result.csv текущего эксперимента
